# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

# zSTD compression

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.37.3
Rasterio version: 1.4.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [3]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [4]:

EVENT_NAME = '202309_Hurricane_Idalia'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'sentinel2'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [5]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}


In [6]:
COG_PROFILE


{'driver': 'COG',
 'bigtiff': 'IF_SAFER',
 'num_threads': 'ALL_CPUS',
 'compress': 'zstd',
 'zstd_level': 22}

## Initialize AWS S3 Client with automatic credential detection

In [7]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

⚠️ S3 client initialized (limited bucket list access)
✅ Confirmed access to nasa-disasters bucket
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 72 .tif files in the S3 bucket.


['drcs_activations/202309_Hurricane_Idalia/sentinel2/pre_event/20230720/cir/S2A_colorInfrared_20230720_160831_T16RGS.tif',
 'drcs_activations/202309_Hurricane_Idalia/sentinel2/pre_event/20230720/cir/S2A_colorInfrared_20230720_160831_T16RGT.tif',
 'drcs_activations/202309_Hurricane_Idalia/sentinel2/pre_event/20230720/cir/S2A_colorInfrared_20230720_160831_T16RGU.tif',
 'drcs_activations/202309_Hurricane_Idalia/sentinel2/pre_event/20230720/cir/S2A_colorInfrared_20230720_160831_T17RKM.tif',
 'drcs_activations/202309_Hurricane_Idalia/sentinel2/pre_event/20230720/cir/S2A_colorInfrared_20230720_160831_T17RKN.tif',
 'drcs_activations/202309_Hurricane_Idalia/sentinel2/pre_event/20230720/cir/S2A_colorInfrared_20230720_160831_T17RKP.tif',
 'drcs_activations/202309_Hurricane_Idalia/sentinel2/pre_event/20230720/cir/S2A_colorInfrared_20230720_160831_T17RLM.tif',
 'drcs_activations/202309_Hurricane_Idalia/sentinel2/pre_event/20230720/cir/S2A_colorInfrared_20230720_160831_T17RLN.tif',
 'drcs_activatio

## Configure bucket and paths (no need to create session manually)

In [8]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

## Define Chunked COG Conversion Function

This function handles the conversion of files to Cloud Optimized GeoTIFFs with:
- Chunked processing to handle large files
- Memory monitoring
- Progress tracking
- Proper CRS and caching

In [9]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 325
  - Total size: 70.52 GB

📁 Cached files (first 10):
  - drcs_activations/20230719_SevereWx_NC/aria/ARIA_DPM_Sentinel-1_North_Carolina_Tornado.tif (4.2 MB)
  - drcs_activations/20230719_SevereWx_NC/aria/ARIA_DPMraw_Sentinel-1_North_Carolina_Tornado.tif (4.2 MB)
  - drcs_activations/20230719_SevereWx_NC/landsat/LC08_L1TP_20230706_015035_colorInfrared.tif (178.9 MB)
  - drcs_activations/20230719_SevereWx_NC/landsat/LC08_L1TP_20230706_015035_naturalColor.tif (178.9 MB)
  - drcs_activations/20230719_SevereWx_NC/landsat/LC08_L1TP_20230706_015035_trueColor.tif (178.9 MB)
  - drcs_activations/20230719_SevereWx_NC/landsat/LC09_L1TP_20230628_015035_colorInfrared.tif (178.9 MB)
  - drcs_activations/20230719_SevereWx_NC/landsat/LC09_L1TP_20230628_015035_naturalColor.tif (178.9 MB)
  - drcs_activations/20230719_SevereWx_NC/landsat/LC09_L1TP_20230628_015035_trueColor.tif (178.9 MB)
  - drcs_activations/20230719_SevereWx_NC/sentinel1

(325, 75720310728)

In [10]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

# Process files

In [11]:
keys

['drcs_activations/202309_Hurricane_Idalia/sentinel2/pre_event/20230720/cir/S2A_colorInfrared_20230720_160831_T16RGS.tif',
 'drcs_activations/202309_Hurricane_Idalia/sentinel2/pre_event/20230720/cir/S2A_colorInfrared_20230720_160831_T16RGT.tif',
 'drcs_activations/202309_Hurricane_Idalia/sentinel2/pre_event/20230720/cir/S2A_colorInfrared_20230720_160831_T16RGU.tif',
 'drcs_activations/202309_Hurricane_Idalia/sentinel2/pre_event/20230720/cir/S2A_colorInfrared_20230720_160831_T17RKM.tif',
 'drcs_activations/202309_Hurricane_Idalia/sentinel2/pre_event/20230720/cir/S2A_colorInfrared_20230720_160831_T17RKN.tif',
 'drcs_activations/202309_Hurricane_Idalia/sentinel2/pre_event/20230720/cir/S2A_colorInfrared_20230720_160831_T17RKP.tif',
 'drcs_activations/202309_Hurricane_Idalia/sentinel2/pre_event/20230720/cir/S2A_colorInfrared_20230720_160831_T17RLM.tif',
 'drcs_activations/202309_Hurricane_Idalia/sentinel2/pre_event/20230720/cir/S2A_colorInfrared_20230720_160831_T17RLN.tif',
 'drcs_activatio

# senintel 2, colorinfrared

In [14]:
# Define filename creator functions for different file types

def create_cog_filename_sentinel2_pre_event(f, EVENT_NAME):
    """Create COG filename for Sentinel-2 earthquake files, moving date to end and capitalizing Color."""
    f2 = Path(f).stem
    parts = f2.split('_')
    
    # Find the date part (YYYYMMDD format)
    date_index = None
    date_str = None
    
    for i, part in enumerate(parts):
        if len(part) == 8 and part.isdigit() and part.startswith('20'):
            date_index = i
            date_str = part
            break
    
    if date_index is not None and date_str:
        # Format date
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
        
        # Process parts, capitalizing "color" in color type names
        processed_parts = []
        for i, part in enumerate(parts):
            if i == date_index:
                continue  # Skip the date
            # Capitalize "color" in truecolorRGB and naturalcolorRGB
            if 'colorRGB' in part:
                part = part.replace('colorRGB', 'ColorRGB')
            processed_parts.append(part)
        
        # Reconstruct with date at end
        cog_filename = f'{EVENT_NAME}_pre_event_{"_".join(processed_parts)}_{formatted_date}_day.tif'
    else:
        # Fallback
        cog_filename = f'{EVENT_NAME}_{f2}.tif'
    
    return cog_filename

filter_str = 'colorInfrared'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_sentinel2_pre_event(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202309_Hurricane_Idalia_pre_event_S2A_colorInfrared_160831_T16RGS_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_colorInfrared_160831_T16RGT_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_colorInfrared_160831_T16RGU_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_colorInfrared_160831_T17RKM_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_colorInfrared_160831_T17RKN_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_colorInfrared_160831_T17RKP_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_colorInfrared_160831_T17RLM_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_colorInfrared_160831_T17RLN_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_colorInfrared_160831_T17RLP_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2B_colorInfrared_155829_T17RLK_2023-07-22_day.tif
  202309_Hurricane_Idalia_pre_event_S2B_colorInfrared_155829_T17RLL_2023-07-22_day.tif
  202309_Hurricane_Ida

In [15]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_sentinel2_pre_event, 
                                target_dir = "Sentinel-2/cir", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202309_Hurricane_Idalia_pre_event_S2A_colorInfrared_160831_T16RGS_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_colorInfrared_160831_T16RGT_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_colorInfrared_160831_T16RGU_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_colorInfrared_160831_T17RKM_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_colorInfrared_160831_T17RKN_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_colorInfrared_160831_T17RKP_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_colorInfrared_160831_T17RLM_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_colorInfrared_160831_T17RLN_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_colorInfrared_160831_T17RLP_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2B_colorInfrared_155829_T17RLK_2023-07-22_day.tif
  202309_Hurricane_Idalia_pre_event_S2B_colorInfrared_155829_T17RLL_2023-07-22_day.tif
  202309_Hurricane_Idali

   [BAND 2/3] Processing...


   [BAND 3/3] Processing...


   [VERIFY] Checking reprojected data...
   [VERIFY] RGB file without nodata - all pixel values are valid
   [VERIFY] Band 1: min=38, max=62, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=46, max=72, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=56, max=80, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpae8730vx_temp.tif



...

   [VERIFY] Checking reprojected data...
   [VERIFY] RGB file without nodata - all pixel values are valid
   [VERIFY] Band 1: min=58, max=130, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=66, max=136, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=74, max=134, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpdv156x70_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp7fk49be7.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-2/cir/202309_Hurricane_Idalia_pre_event_S2B_colorInfrared_155829_T17RNN_2023-07-22_day.tif
   [MEMORY] Final: 1664.9 MB (Change: +5.1 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202309_Hurricane_Idalia_pre_event_S2B_colorInfrared_155829_T17RNN_2023-07-22_day.tif

✅ Batch processing complete: 24 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Sentinel-2/cir/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Sentinel-2/cir/files_converted.csv
📁 COGs saved locally to: output/202309_Hurricane_Idalia

📊 BATCH PROCESSING SUMMARY
Total files processed: 24
Successful: 24
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-09T01:31:15

In [16]:
keys

['drcs_activations/202309_Hurricane_Idalia/sentinel2/pre_event/20230720/cir/S2A_colorInfrared_20230720_160831_T16RGS.tif',
 'drcs_activations/202309_Hurricane_Idalia/sentinel2/pre_event/20230720/cir/S2A_colorInfrared_20230720_160831_T16RGT.tif',
 'drcs_activations/202309_Hurricane_Idalia/sentinel2/pre_event/20230720/cir/S2A_colorInfrared_20230720_160831_T16RGU.tif',
 'drcs_activations/202309_Hurricane_Idalia/sentinel2/pre_event/20230720/cir/S2A_colorInfrared_20230720_160831_T17RKM.tif',
 'drcs_activations/202309_Hurricane_Idalia/sentinel2/pre_event/20230720/cir/S2A_colorInfrared_20230720_160831_T17RKN.tif',
 'drcs_activations/202309_Hurricane_Idalia/sentinel2/pre_event/20230720/cir/S2A_colorInfrared_20230720_160831_T17RKP.tif',
 'drcs_activations/202309_Hurricane_Idalia/sentinel2/pre_event/20230720/cir/S2A_colorInfrared_20230720_160831_T17RLM.tif',
 'drcs_activations/202309_Hurricane_Idalia/sentinel2/pre_event/20230720/cir/S2A_colorInfrared_20230720_160831_T17RLN.tif',
 'drcs_activatio

# sentinel 2, natural color

In [18]:
# Define filename creator functions for different file types


filter_str = 'natural'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_sentinel2_pre_event(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202309_Hurricane_Idalia_pre_event_S2A_naturalColor_160831_T16RGS_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_naturalColor_160831_T16RGT_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_naturalColor_160831_T16RGU_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_naturalColor_160831_T17RKM_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_naturalColor_160831_T17RKN_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_naturalColor_160831_T17RKP_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_naturalColor_160831_T17RLM_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_naturalColor_160831_T17RLN_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_naturalColor_160831_T17RLP_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2B_naturalColor_155829_T17RLK_2023-07-22_day.tif
  202309_Hurricane_Idalia_pre_event_S2B_naturalColor_155829_T17RLL_2023-07-22_day.tif
  202309_Hurricane_Idalia_pre_eve

In [19]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_sentinel2_pre_event, 
                                target_dir = "Sentinel-2/natural", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202309_Hurricane_Idalia_pre_event_S2A_naturalColor_160831_T16RGS_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_naturalColor_160831_T16RGT_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_naturalColor_160831_T16RGU_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_naturalColor_160831_T17RKM_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_naturalColor_160831_T17RKN_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_naturalColor_160831_T17RKP_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_naturalColor_160831_T17RLM_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_naturalColor_160831_T17RLN_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_naturalColor_160831_T17RLP_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2B_naturalColor_155829_T17RLK_2023-07-22_day.tif
  202309_Hurricane_Idalia_pre_event_S2B_naturalColor_155829_T17RLL_2023-07-22_day.tif
  202309_Hurricane_Idalia_pre_event

   [BAND 2/3] Processing...


   [BAND 3/3] Processing...


   [VERIFY] Checking reprojected data...
   [VERIFY] RGB file without nodata - all pixel values are valid
   [VERIFY] Band 1: min=32, max=142, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=38, max=144, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=44, max=140, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpaif7siw0_temp.tif



...

   [VERIFY] Checking reprojected data...
   [VERIFY] RGB file without nodata - all pixel values are valid
   [VERIFY] Band 1: min=50, max=174, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=58, max=156, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=64, max=156, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmplzd6ibsr_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpbot60tpq.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-2/natural/202309_Hurricane_Idalia_pre_event_S2B_naturalColor_155829_T17RNN_2023-07-22_day.tif
   [MEMORY] Final: 1671.4 MB (Change: +0.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202309_Hurricane_Idalia_pre_event_S2B_naturalColor_155829_T17RNN_2023-07-22_day.tif

✅ Batch processing complete: 24 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Sentinel-2/natural/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Sentinel-2/natural/files_converted.csv
📁 COGs saved locally to: output/202309_Hurricane_Idalia

📊 BATCH PROCESSING SUMMARY
Total files processed: 24
Successful: 24
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-0

# sentinel 2, shortwave Infrared

In [20]:


filter_str = 'shortwaveInfrared'
target_dir = "Sentinel-2/swir"

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_sentinel2_pre_event(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202309_Hurricane_Idalia_pre_event_S2A_shortwaveInfrared_160831_T16RGS_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_shortwaveInfrared_160831_T16RGT_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_shortwaveInfrared_160831_T16RGU_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_shortwaveInfrared_160831_T17RKM_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_shortwaveInfrared_160831_T17RKN_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_shortwaveInfrared_160831_T17RKP_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_shortwaveInfrared_160831_T17RLM_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_shortwaveInfrared_160831_T17RLN_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_shortwaveInfrared_160831_T17RLP_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2B_shortwaveInfrared_155829_T17RLK_2023-07-22_day.tif
  202309_Hurricane_Idalia_pre_event_S2B_shortwaveInfrared_155829_T17R

In [21]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_sentinel2_pre_event, 
                                target_dir = target_dir, 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202309_Hurricane_Idalia_pre_event_S2A_shortwaveInfrared_160831_T16RGS_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_shortwaveInfrared_160831_T16RGT_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_shortwaveInfrared_160831_T16RGU_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_shortwaveInfrared_160831_T17RKM_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_shortwaveInfrared_160831_T17RKN_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_shortwaveInfrared_160831_T17RKP_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_shortwaveInfrared_160831_T17RLM_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_shortwaveInfrared_160831_T17RLN_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2A_shortwaveInfrared_160831_T17RLP_2023-07-20_day.tif
  202309_Hurricane_Idalia_pre_event_S2B_shortwaveInfrared_155829_T17RLK_2023-07-22_day.tif
  202309_Hurricane_Idalia_pre_event_S2B_shortwaveInfrared_155829_T17RLL

   [BAND 2/3] Processing...


   [BAND 3/3] Processing...


   [VERIFY] Checking reprojected data...
   [VERIFY] RGB file without nodata - all pixel values are valid
   [VERIFY] Band 1: min=32, max=140, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=38, max=130, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=44, max=140, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmp18kzubut_temp.tif



...

   [VERIFY] Checking reprojected data...
   [VERIFY] RGB file without nodata - all pixel values are valid
   [VERIFY] Band 1: min=44, max=160, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=54, max=154, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=64, max=156, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpcsf1pfbs_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp532a8tim.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-2/swir/202309_Hurricane_Idalia_pre_event_S2B_shortwaveInfrared_155829_T17RNN_2023-07-22_day.tif
   [MEMORY] Final: 1671.6 MB (Change: +0.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202309_Hurricane_Idalia_pre_event_S2B_shortwaveInfrared_155829_T17RNN_2023-07-22_day.tif

✅ Batch processing complete: 24 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Sentinel-2/swir/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Sentinel-2/swir/files_converted.csv
📁 COGs saved locally to: output/202309_Hurricane_Idalia

📊 BATCH PROCESSING SUMMARY
Total files processed: 24
Successful: 24
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-

## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.

## Memory Usage Summary

You can check the final memory usage and cleanup

In [22]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")


📊 Memory Usage Summary:
  Current memory usage: 1671.6 MB
  Available memory: 24712.0 MB
  Memory percent used: 21.9%
